In [42]:
import pandas as pd

In [43]:
df_electric = pd.read_csv(r'C:\Users\ADMIN\Downloads\MODEL_TFT\data\raw\ember_data\all_country_data.csv')
df_weather = pd.read_csv(r'C:\Users\ADMIN\Downloads\MODEL_TFT\data\processed\All_weather_data\all_country_20.csv', sep=';')

In [44]:
df_electric.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 31640 entries, 0 to 31639
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   entity          31640 non-null  object 
 1   entity code     31640 non-null  object 
 2   date            31640 non-null  object 
 3   series          31640 non-null  object 
 4   generation_TWh  31640 non-null  float64
dtypes: float64(1), object(4)
memory usage: 1.2+ MB


In [45]:
df_electric.head(30)

,entity,entity code,date,series,generation_TWh
0,Australia,AUS,2018-01-01,Demand,19.96
1,Australia,AUS,2018-01-01,Clean,3.27
2,Australia,AUS,2018-01-01,Fossil,16.69
3,Australia,AUS,2018-01-01,Gas and Other Fossil,2.53
4,Australia,AUS,2018-01-01,"Hydro, Bioenergy and Other Renewables",1.12
5,Australia,AUS,2018-01-01,Renewables,3.27
6,Australia,AUS,2018-01-01,Wind and Solar,2.15
7,Australia,AUS,2018-01-01,Bioenergy,0.03
8,Australia,AUS,2018-01-01,Coal,14.16
9,Australia,AUS,2018-01-01,Gas,2.53


In [46]:
df_weather.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2016 entries, 0 to 2015
Data columns (total 6 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   date           2016 non-null   object 
 1   precipitation  2016 non-null   float64
 2   solar          2016 non-null   float64
 3   humidity       2016 non-null   float64
 4   temperature    2016 non-null   float64
 5   area           2016 non-null   object 
dtypes: float64(4), object(2)
memory usage: 94.6+ KB


In [47]:
df_weather.head()

,date,precipitation,solar,humidity,temperature,area
0,1/1/2018,1.281013,1.344069,90.716614,-1.162657,Romania
1,2/1/2018,2.159927,1.724244,89.058387,-2.042290,Romania
2,3/1/2018,2.629274,2.723513,89.057320,1.208301,Romania
3,4/1/2018,0.858431,5.288260,75.713122,13.300846,Romania
4,5/1/2018,1.999852,6.092853,70.209946,17.078283,Romania


In [48]:


# 1. Chuyển đổi cột 'date' sang kiểu datetime (báo cho pandas biết gốc là MM/DD/YYYY)
df_weather['date'] = pd.to_datetime(df_weather['date'])

# # 2. Chuyển ngược lại thành chuỗi theo định dạng DD/MM/YYYY
# df_weather['date'] = df_weather['date'].dt.to_period('M').dt.to_timestamp()

# df_weather['date'] = pd.to_datetime(df_weather['date'], format="%d/%m/%Y")

# Kiểm tra kết quả
print(df_weather.head())

        date  precipitation     solar   humidity  temperature     area
0 2018-01-01       1.281013  1.344069  90.716614    -1.162657  Romania
1 2018-02-01       2.159927  1.724244  89.058387    -2.042290  Romania
2 2018-03-01       2.629274  2.723513  89.057320     1.208301  Romania
3 2018-04-01       0.858431  5.288260  75.713122    13.300846  Romania
4 2018-05-01       1.999852  6.092853  70.209946    17.078283  Romania


In [49]:
# Trước tiên, đảm bảo cột 'date' của df_electric có kiểu datetime giống với df_weather
df_electric['date'] = pd.to_datetime(df_electric['date'])

# Thực hiện merge 2 dataframe
# Sử dụng how='left' nhằm đảm bảo giữ nguyên toàn bộ số dòng của bảng điện lưới (df_electric)
df_merged = pd.merge(
    df_electric,
    df_weather,
    left_on=['entity', 'date'],
    right_on=['area', 'date'],
    how='left'
)

# (Tuỳ chọn) Cột 'area' sau khi merge sẽ bị thừa (do nó giống 'entity'), nên có thể xóa đi
if 'area' in df_merged.columns:
    df_merged = df_merged.drop(columns=['area'])

# Kiểm tra thử kết quả
df_merged.tail()


,entity,entity code,date,series,generation_TWh,precipitation,solar,humidity,temperature
31635,Viet Nam,VNM,2025-10-01,Other Fossil,0.00,10.963779,3.963385,88.290895,23.947788
31636,Viet Nam,VNM,2025-10-01,Solar,1.20,10.963779,3.963385,88.290895,23.947788
31637,Viet Nam,VNM,2025-10-01,Wind,0.84,10.963779,3.963385,88.290895,23.947788
31638,Viet Nam,VNM,2025-10-01,Total Generation,26.19,10.963779,3.963385,88.290895,23.947788
31639,Viet Nam,VNM,2025-10-01,Net Imports,0.00,10.963779,3.963385,88.290895,23.947788


In [50]:
# Đảm bảo cột date có cùng định dạng
df_electric['date'] = pd.to_datetime(df_electric['date'])

# Chỉnh lại tên quốc gia bị lệch trong df_electric cho khớp với df_weather
df_electric['entity'] = df_electric['entity'].replace({'Philippines (the)': 'Philippines'})

# Lọc bỏ quốc gia 'Morocco' ra khỏi df_electric (vì df_weather không có dữ liệu để ghép)
df_electric = df_electric[df_electric['entity'] != 'Morocco']

# Thực hiện merge 2 dataframe
df_merged = pd.merge(
    df_electric,
    df_weather,
    left_on=['entity', 'date'],
    right_on=['area', 'date'],
    how='left'
)

# Cột 'area' sau khi merge sẽ bị thừa (vì giống 'entity'), nên xóa để gọn dữ liệu
if 'area' in df_merged.columns:
    df_merged = df_merged.drop(columns=['area'])

# Kiểm tra lại kết quả tổng quát
df_merged.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30290 entries, 0 to 30289
Data columns (total 9 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   entity          30290 non-null  object        
 1   entity code     30290 non-null  object        
 2   date            30290 non-null  datetime64[ns]
 3   series          30290 non-null  object        
 4   generation_TWh  30290 non-null  float64       
 5   precipitation   30290 non-null  float64       
 6   solar           30290 non-null  float64       
 7   humidity        30290 non-null  float64       
 8   temperature     30290 non-null  float64       
dtypes: datetime64[ns](1), float64(5), object(3)
memory usage: 2.1+ MB


In [51]:
df_merged.head(30)

,entity,entity code,date,series,generation_TWh,precipitation,solar,humidity,temperature
0,Australia,AUS,2018-01-01,Demand,19.96,2.944177,7.184404,45.105328,29.743529
1,Australia,AUS,2018-01-01,Clean,3.27,2.944177,7.184404,45.105328,29.743529
2,Australia,AUS,2018-01-01,Fossil,16.69,2.944177,7.184404,45.105328,29.743529
3,Australia,AUS,2018-01-01,Gas and Other Fossil,2.53,2.944177,7.184404,45.105328,29.743529
4,Australia,AUS,2018-01-01,"Hydro, Bioenergy and Other Renewables",1.12,2.944177,7.184404,45.105328,29.743529
5,Australia,AUS,2018-01-01,Renewables,3.27,2.944177,7.184404,45.105328,29.743529
6,Australia,AUS,2018-01-01,Wind and Solar,2.15,2.944177,7.184404,45.105328,29.743529
7,Australia,AUS,2018-01-01,Bioenergy,0.03,2.944177,7.184404,45.105328,29.743529
8,Australia,AUS,2018-01-01,Coal,14.16,2.944177,7.184404,45.105328,29.743529
9,Australia,AUS,2018-01-01,Gas,2.53,2.944177,7.184404,45.105328,29.743529


In [52]:
df_electric['entity'].unique()

array(['Australia', 'Austria', 'Chile', 'Colombia', 'Czechia', 'France',
       'Germany', 'Kazakhstan', 'Malaysia', 'Pakistan', 'Philippines',
       'Poland', 'Serbia', 'South Africa', 'Spain', 'Sweden', 'Taiwan',
       'Turkey', 'Ukraine', 'Viet Nam'], dtype=object)

In [53]:
# Lưu dữ liệu ra một file CSV mới
df_merged.to_csv('merged_electric_weather_data.csv', index=False)

print("Đã lưu dữ liệu thành công!")


Đã lưu dữ liệu thành công!
